# ULDES requirement in Germany by 2030+: results on storage capacity, electricity mix, and electricity balance

N.B.: this code is human-generated and improved and assisted by AI.

This notebook reads the networks solved by PyPSA-Eur in the context of the [noon-energy-storage project](https://github.com/open-energy-transition/noon-energy-storage).
It produces, for every scenario:

1. **Static maps** - the energy capacity of the storage technologies and the electricity mix,
   drawn as one pie per bus of the model, plus summary tables and system indicators.
2. **Time series** - the electricity balance of a country over selected periods of the year,
   which shows how demand is met hour by hour and how storage is used.

**Before running it**

- Install the environment of the repository (`pixi install`) and select it as the kernel of this notebook.
- Make sure the scenarios listed in section 1.1 have been solved, i.e. that the folder
  `results/<run>/<scenario>/networks/` contains the solved network. Scenarios that are missing are
  reported when loading and simply skipped.

**How it is organised**

Everything that can be changed - which results to read, which technologies to show, colours and
periods - lives in section 1 of this notebook. The calculations and the plots live in
`analysis_functions.py`, next to this notebook; open that file to see how a number is computed.
Run the cells in order: section 2 and section 3 are independent of each other.

## 1. Setup

In [ ]:
# Re-run this cell after editing analysis_functions.py, to work with the version of
# the file as it is on disk. Both are needed: reload() replaces the module that Python
# cached when it was first imported, autoreload picks up any later edit on its own.
%load_ext autoreload
%autoreload 2

import importlib

import analysis_functions
import matplotlib.pyplot as plt
import pandas as pd

importlib.reload(analysis_functions)  # always start from the file as it is on disk

from analysis_functions import (  # noqa: E402
    ELECTRICITY_MIX,
    # column names of the result tables
    ENERGY_CAPACITY,
    get_electricity_balance,
    get_electricity_mix,
    # numbers
    get_storage_capacity,
    kpi_table,
    # reading the results
    load_networks,
    plot_balance,
    # plots
    plot_map,
    summary_table,
)

# Show every column and row of a table, instead of collapsing the middle ones into "...".
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

# Look and feel of every figure of the notebook.
plt.rcParams.update(
    {
        "font.family": "sans-serif",
        "font.size": 12,
        "axes.titlesize": 16,
        "axes.titleweight": "bold",
        "axes.labelsize": 14,
        "xtick.labelsize": 11,
        "ytick.labelsize": 11,
        "legend.fontsize": 10,
        "legend.title_fontsize": 10,
        "legend.frameon": True,
        "legend.edgecolor": "black",
        "axes.grid": True,
        "grid.alpha": 0.3,
        "figure.autolayout": True,
    }
)

### 1.1 Which results to analyse

A *run* is one folder of results, produced by one Snakemake execution, and it contains one
*scenario* per folder inside it. The scenario name ends with its planning year, which is used to
find the file of the solved network.

In [ ]:
RESULTS_DIR = "../results"  # where Snakemake wrote the results
RUN = "52buses-3h-final"  # folder of the run to analyse among 52buses-3h-final, 52buses-3h, bz-3h
CLUSTERS = "52"  # number of buses of the run, as it appears in the file names
# if run = bz-3h, choose clusters = "adm"

# The scenarios of config/scenarios.noon.yaml. Comment out the ones that are not needed:
# every scenario adds a few minutes and a few GB of memory.
SCENARIOS = [
    # technology
    "cy2021-base-2030",  # reference: no mds, no res
    "cy2021-lds-2030",  # long-duration storage: mds + res
    "cy2021-lds-store-2030",  # res modelled as a store with charger and discharger
    # weather year
    "cy2023-lds-2030",  # same as cy2021-lds-2030, with the weather of 2023
    # foresight
    "cy2021-lds-brownfield-2030",  # myopic: today's fleet is kept
    # planning horizon, greenfield
    "cy2021-lds-2035",
    "cy2021-lds-2040",
    "cy2021-lds-2050",
    # planning horizon, brownfield
    "cy2021-lds-brownfield-2035",
    "cy2021-lds-brownfield-2040",
    "cy2021-lds-brownfield-2050",
]

COUNTRY = "DE"  # country focus
OUTPUT_DIR = RUN  # folder where the figures are saved

### 1.2 Technologies

Each of these settings maps the carriers used inside the network to the names and the groups shown
in the plots and the tables. Carriers that are not part of a scenario are ignored, so the same
settings work for every scenario.

In [ ]:
# Storage technologies to report: carrier in the network -> (name to show, group in the tables).
# The group puts the durations of a technology on a single row of the summary tables.
STORAGE_TECHNOLOGIES = {
    "li-ion 6h": ("li-ion 6h", "li-ion"),
    "li-ion 24h": ("li-ion 24h", "li-ion"),
    "home battery": ("home battery", "li-ion"),
    "lfp": ("lfp 6h", "other"),
    "vanadium": ("vanadium 10h", "other"),
    "lair": ("lair 12h", "other"),
    "pair 24h": ("pair 24h", "pair"),
    "pair 100h": ("pair 100h", "pair"),
    "mds": ("mds 100h", "mds"),
    "res": ("res", "res"),
    "res 24h": ("res 24h", "res"),
    "res 100h": ("res 100h", "res"),
    "res 300h": ("res 300h", "res"),
    "res 500h": ("res 500h", "res"),
    "PHS": ("phs", "phs"),
}

# Storage technologies whose store does not hold electricity: a "res" store holds a carbon
# medium, so its capacity is converted into electricity using the efficiency of its
# discharger. Home batteries store electricity directly and are not listed here.
NON_ELECTRIC_STORES = ["res"]

# Technologies that generate electricity, grouped into the categories shown on the maps.
# Anything not listed here is left out of the electricity mix. "AC" and "DC" are the flows on the
# transmission grid, i.e. what a bus imports from or exports to its neighbours.
ELECTRICITY_CARRIERS = {
    "nuclear": "nuclear",
    "onwind": "onwind",
    "offwind-ac": "offwind",
    "offwind-dc": "offwind",
    "offwind-float": "offwind",
    "solar": "solar",
    "solar rooftop": "solar",
    "solar-hsat": "solar",
    "CCGT": "fossil fuels",
    "OCGT": "fossil fuels",
    "urban central gas CHP": "fossil fuels",
    "urban central gas CHP CC": "fossil fuels",
    "coal": "fossil fuels",
    "lignite": "fossil fuels",
    "oil": "fossil fuels",
    "H2 Fuel Cell": "low-carbon fuels",
    "OCGT methanol": "low-carbon fuels",
    "urban central solid biomass CHP": "low-carbon fuels",
    "urban central solid biomass CHP CC": "low-carbon fuels",
    "ror": "hydro",
    "hydro": "hydro",
    "AC": "net import",
    "DC": "net import",
}

# Technologies that consume electricity, shown in the electricity balance over time. Everything
# that is demand rather than storage is gathered under "gross consumption", which is drawn as a
# line. Listed after the generation ones, so that "H2 Fuel Cell" appears as consumption here.
CONSUMPTION_CARRIERS = {
    "electricity": "gross consumption",
    "electricity distribution grid": "gross consumption",
    "DAC": "gross consumption",
    "Haber-Bosch": "gross consumption",
    "rural air heat pump": "gross consumption",
    "rural ground heat pump": "gross consumption",
    "rural resistive heater": "gross consumption",
    "urban central air heat pump": "gross consumption",
    "urban central resistive heater": "gross consumption",
    "urban decentral air heat pump": "gross consumption",
    "urban decentral resistive heater": "gross consumption",
    "load": "load shedding",
    "H2 Electrolysis": "H2 electrolysis",
    "H2 Fuel Cell": "H2 fuel cell",
    "BEV charger": "BEV charger",
    "V2G": "vehicle-to-grid",
    "res charger": "res charger",
    "res discharger": "res discharger",
    "home battery charger": "home battery charger",
    "home battery discharger": "home battery discharger",
}

# Carriers whose investment cost and curtailment are reported among the system indicators.
INVESTMENT_CARRIERS = [
    "res",
    "res charger",
    "res discharger",
    "res 24h",
    "res 100h",
    "res 300h",
    "res 500h",
]
RENEWABLE_CARRIERS = [
    "solar",
    "solar-hsat",
    "solar rooftop",
    "onwind",
    "offwind-ac",
    "offwind-dc",
    "offwind-float",
]

# The settings above are read by the functions in the two forms below. The entries
# "lfp": ("lfp 6h", "other") and "PHS": ("phs", "phs") of STORAGE_TECHNOLOGIES, say,
# end up as:
#
#   STORAGE_LABELS = {..., "lfp": "lfp 6h",    "PHS": "phs", ...}   carrier -> name shown
#   STORAGE_GROUPS = {..., "lfp 6h": "other",  "phs": "phs", ...}   name shown -> group
#
# so a map shows a slice named "lfp 6h" while the summary tables add it to the row
# "other". Print them to see the whole list.
STORAGE_LABELS = {carrier: name for carrier, (name, _) in STORAGE_TECHNOLOGIES.items()}
STORAGE_GROUPS = {name: group for name, group in STORAGE_TECHNOLOGIES.values()}

# Carriers of the electricity balance: the generation and consumption ones, plus the
# storage technologies under the name they are shown with.
BALANCE_CARRIERS = {**ELECTRICITY_CARRIERS, **STORAGE_LABELS, **CONSUMPTION_CARRIERS}

# Listed under their own heading in the legend of the electricity balance.
STORAGE_AND_EVS = set(STORAGE_LABELS.values()) | {
    "res charger",
    "res discharger",
    "home battery charger",
    "home battery discharger",
    "BEV charger",
    "vehicle-to-grid",
}

### 1.3 Colours and hatches

One colour per technology, used by every figure. The order of this setting is also the order of
the legends and the stacking order of the time series, so keep the technologies that should be
read together next to each other. Hatches tell apart technologies drawn in the same colour, such
as charging and discharging.

In [ ]:
CARRIER_COLORS = {
    # generation
    "fossil fuels": "#111111",
    "low-carbon fuels": "#6c5d28",
    "nuclear": "#ff8c00",
    "hydro": "#298c81",
    "onwind": "#235ebc",
    "offwind": "#6895dd",
    "solar": "#FFD900",
    "net import": "#FF0000FF",
    "load shedding": "#797878",
    "H2 fuel cell": "#006707FF",
    "H2 electrolysis": "#006707FF",
    # storage, from the longest to the shortest duration
    "phs": "#298c81",
    "res": "#ff4d00",
    "res charger": "#ff4d00",
    "res discharger": "#ff4d00",
    "res 500h": "#ff4d00",
    "res 300h": "#ff4d00",
    "res 100h": "#ff4d00",
    "mds 100h": "#edba1c",
    "pair 100h": "#87CEEB",
    "BEV charger": "#FF00FF",
    "vehicle-to-grid": "#FF00FF",
    "res 24h": "#ff4d00",
    "pair 24h": "#87CEEB",
    "li-ion 24h": "#6db85b",
    "lair 12h": "#005D81",
    "vanadium 10h": "#9B111E",
    "li-ion 6h": "#6db85b",
    "lfp 6h": "#d4ed6a",
    "home battery charger": "#6db85b",
    "home battery discharger": "#6db85b",
    "home battery": "#6db85b",
}

CARRIER_HATCHES = {
    "phs": "///",
    "res 100h": "---",
    "res 300h": "|||",
    "res 500h": "...",
    "res charger": "///",
    "res discharger": "...",
    "pair 100h": "---",
    "li-ion 6h": "///",
    "home battery": "...",
    "home battery charger": "///",
    "home battery discharger": "...",
    "BEV charger": "///",
    "H2 electrolysis": "///",
}

### 1.4 Load the networks

Reading the networks takes a few minutes and a few GB of memory: they stay in memory for the rest
of the notebook, so this cell only has to be run once.

In [ ]:
networks = load_networks(RUN, SCENARIOS, results_dir=RESULTS_DIR, clusters=CLUSTERS)

## 2. Static maps

The two maps below summarise a whole year of operation of a scenario on a single picture. Each pie
is one bus of the model, its size is the total of the quantity plotted, and its slices are the
technologies. The table in the corner repeats the totals for Europe and for the selected country,
and the legend gives the scale of the pies.

The next cell computes the two tables the maps are drawn from. Both are indexed by bus and carrier
and can be inspected directly, for instance with `storage_capacity["cy2021-lds-2030"].head()`.

In [ ]:
storage_capacity = {
    scenario: get_storage_capacity(n, STORAGE_LABELS, NON_ELECTRIC_STORES)
    for scenario, n in networks.items()
}

electricity_mix = {
    scenario: get_electricity_mix(n, ELECTRICITY_CARRIERS)
    for scenario, n in networks.items()
}

### 2.1 Storage energy capacity

In [ ]:
for scenario, n in networks.items():
    plot_map(
        n,
        storage_capacity[scenario],
        value_col=ENERGY_CAPACITY,
        colors=CARRIER_COLORS,
        hatches=CARRIER_HATCHES,
        groups=STORAGE_GROUPS,
        country=COUNTRY,
        title="Energy storage capacity",
        target_ratio=4.5,
        output_dir=OUTPUT_DIR,
    )

### 2.2 Electricity mix

How much electricity each technology generates over the year, in TWh. "net import" is the
electricity a bus takes from its neighbours; it is negative, and therefore not drawn, for a bus
that exports more than it imports; in the summary table, it is negative when considering all the countries as it accounts for grid losses.

In [ ]:
for scenario, n in networks.items():
    plot_map(
        n,
        electricity_mix[scenario],
        value_col=ELECTRICITY_MIX,
        colors=CARRIER_COLORS,
        country=COUNTRY,
        title="Electricity mix",
        target_ratio=1.3,
        labelspacing_circles=1,
        output_dir=OUTPUT_DIR,
    )

### 2.3 Summary tables

The numbers behind the maps, with all the scenarios side by side - a single scenario included.
Add `country=COUNTRY` to restrict a table to one country, and replace `ENERGY_CAPACITY` with
`POWER_CAPACITY` to read power instead of energy. To look at one scenario on its own, with
both columns at once, call `summary_by_carrier(networks[scenario], storage_capacity[scenario],
groups=STORAGE_GROUPS)`.

In [ ]:
storage_summary = summary_table(
    networks, storage_capacity, ENERGY_CAPACITY, groups=STORAGE_GROUPS
)

mix_summary = summary_table(networks, electricity_mix, ELECTRICITY_MIX)

print("Storage energy capacity (GWh)")
display(storage_summary.round(0))

print("Electricity mix (TWh)")
display(mix_summary.round(0))

### 2.4 System indicators

The indicators of every scenario, for Europe and for the selected country: the investments in the RES storage technology, the curtailment of wind and solar as a
share of the total production of the system, and the average marginal price of electricity. The docstring
of `get_kpis` gives the exact formulas.

In [ ]:
kpis = kpi_table(
    networks,
    ELECTRICITY_CARRIERS,
    INVESTMENT_CARRIERS,
    RENEWABLE_CARRIERS,
    country=COUNTRY,
)

display(kpis.round(1))

## 3. Time series

The electricity balance shows, hour by hour, how the demand of a country is met. Areas above zero
feed the grid - generation, storage discharging, imports - and areas below zero withdraw from it -
storage charging, electrolysis, electric vehicles, exports. The dashed line is the gross
consumption the areas have to meet at every hour.

Two short periods are currently used, but any period of the year can be used; the year
itself is ignored, so the same setting works for every weather year.

In [ ]:
PERIODS = {
    "January": ("01-01", "01-10"),
    "July": ("07-01", "07-10"),
}

for scenario, n in networks.items():
    balance = get_electricity_balance(n, COUNTRY, BALANCE_CARRIERS)

    plot_balance(
        balance,
        PERIODS,
        country=COUNTRY,
        name=n.name,
        colors=CARRIER_COLORS,
        hatches=CARRIER_HATCHES,
        storage_carriers=STORAGE_AND_EVS,
        output_dir=OUTPUT_DIR,
    )